# EcoRiego AI — Entrenamiento Compatible con EloquentTinyML

**IMPORTANTE:** Este notebook genera un `modelo_entrenado.h` compatible con la librería `EloquentTinyML` usada en el ESP32.

La diferencia clave con el modelo RL anterior:
- ❌ Sin capas `Sigmoid` (no soportada por EloquentTinyML)
- ❌ Sin capas `Lambda` (no soportada por TFLite Micro)
- ✅ Solo capas `Dense` con `relu` y salida lineal
- ✅ Supervised Learning sobre datos sintéticos agronómicos

### Flujo
```
PASO 1: Generar 20,000 datos sintéticos con física agrícola real
PASO 2: Entrenar red neuronal compatible (Dense + ReLU)
PASO 3: Convertir a TFLite con optimización DEFAULT
PASO 4: Generar modelo_entrenado.h → copiar a lib/Logic/
```

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split
import math

print(f'TensorFlow v{tf.__version__}')
print('Librerias cargadas correctamente.')

## PASO 1 — Generación de Datos Sintéticos Agronómicos

In [ ]:
np.random.seed(42)
CANTIDAD = 20000

# Perfiles de cultivo (FAO-56)
CULTIVOS = {
    'tomate':  {'umbral_riego': 45, 'umbral_optimo': 65, 'kc': 1.05},
    'lechuga': {'umbral_riego': 55, 'umbral_optimo': 70, 'kc': 0.90},
    'papa':    {'umbral_riego': 40, 'umbral_optimo': 60, 'kc': 1.10},
}

FASES = {
    'germinacion': 0.50,
    'vegetativa':  1.00,
    'floracion':   1.20,
    'maduracion':  0.80,
}

def temp_circadiana(hora, mes):
    base = 18.0
    amp  = 10.0
    adj  = 4.0 if 4 <= mes <= 10 else -2.0
    var  = amp * math.sin(math.pi * (hora-6)/12) if 6 <= hora <= 18 else -amp*0.5
    return float(np.clip(base + adj + var + np.random.normal(0, 1.5), 8.0, 38.0))

def hum_aire(temp, mes):
    base = 75.0 if mes in [11,12,1,2,3] else 55.0
    return float(np.clip(base - 1.5*max(0, temp-20) + np.random.normal(0, 5.0), 20.0, 98.0))

def et_ref(temp, hum, hora):
    fs  = max(0, math.sin(math.pi*(hora-6)/12)) if 6<=hora<=18 else 0.0
    vpd = (1 - hum/100.0) * 0.611 * math.exp(17.27*temp/(temp+237.3))
    return float(np.clip(0.408*fs*0.5 + 0.066*vpd*2.0, 0.0, 2.5))

def calcular_riego(suelo, temp, hum, hora, cultivo, fase):
    p = CULTIVOS[cultivo]
    f = FASES[fase]
    
    if suelo >= p['umbral_optimo']: return 0.0
    if hora < 6 or hora >= 23:      return 0.0
    if hum > 88.0 and temp < 18.0:  return 0.0
    
    deficit = p['umbral_riego'] - suelo
    if deficit <= 0: return 0.0
    
    tiempo = deficit * 0.12 * p['kc']
    et = et_ref(temp, hum, hora)
    tiempo += et * f * 1.5
    
    if temp > 30.0:  tiempo *= 1.20
    elif temp < 12.0: tiempo *= 0.75
    if hum < 35.0:  tiempo *= 1.15
    
    tiempo += np.random.normal(0, 0.15)
    return max(0.0, round(float(np.clip(tiempo, 0.0, 15.0)), 2))

# Generacion
lista_cultivos = list(CULTIVOS.keys())
lista_fases    = list(FASES.keys())
registros = []

for i in range(CANTIDAD):
    mes   = np.random.randint(1, 13)
    hora  = round(np.random.uniform(0, 23.99), 2)
    cult  = np.random.choice(lista_cultivos)
    fase  = np.random.choice(lista_fases, p=[0.10, 0.40, 0.30, 0.20])
    temp  = temp_circadiana(hora, mes)
    hum   = hum_aire(temp, mes)
    
    if np.random.random() < 0.40:
        suelo = float(np.random.beta(2,5) * 55)
    else:
        suelo = float(np.random.beta(5,2) * 55 + 45)
    suelo = float(np.clip(round(suelo, 2), 0.0, 100.0))
    
    tiempo = calcular_riego(suelo, temp, hum, hora, cult, fase)
    registros.append({'HumedadSuelo': suelo, 'TemperaturaAire': round(temp,2),
                      'HumedadAire': round(hum,2), 'TiempoRiego': tiempo})

df = pd.DataFrame(registros)
print(f'Dataset generado: {len(df):,} filas')
print(f'Con riego   : {(df.TiempoRiego > 0).sum():,} ({(df.TiempoRiego > 0).mean()*100:.1f}%)')
print(f'Sin riego   : {(df.TiempoRiego == 0).sum():,} ({(df.TiempoRiego == 0).mean()*100:.1f}%)')
print()
df.describe().round(2)

## PASO 2 — Entrenar Red Neuronal Compatible

Arquitectura: `3 → 16 → 8 → 1`  
- Activaciones: solo `relu`  
- Salida: **lineal** (sin sigmoid, sin softmax)  
- Compatible 100% con EloquentTinyML

In [ ]:
X = df[['HumedadSuelo', 'TemperaturaAire', 'HumedadAire']].values
y = df['TiempoRiego'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# RED NEURONAL — Solo Dense + ReLU + salida lineal
# SIN Lambda, SIN Sigmoid, SIN BatchNorm
model = keras.Sequential([
    keras.layers.Input(shape=(3,)),
    keras.layers.Dense(16, activation='relu'),
    keras.layers.Dense(8,  activation='relu'),
    keras.layers.Dense(1)   # Salida LINEAL — critico para compatibilidad
], name='ecoriego_compatible')

model.compile(optimizer='adam', loss='mse', metrics=['mae'])
model.summary()

print('\nEntrenando...')
history = model.fit(
    X_train, y_train,
    epochs=150,
    batch_size=64,
    validation_split=0.1,
    verbose=0
)

loss, mae = model.evaluate(X_test, y_test, verbose=0)
print(f'Entrenamiento finalizado.')
print(f'Loss (MSE) en test : {loss:.4f}')
print(f'MAE en test        : {mae:.4f} minutos de error promedio')

In [ ]:
# Validacion rapida con casos conocidos
print('=' * 55)
print('  VALIDACION — Casos de prueba')
print('=' * 55)
print(f'  {"Caso":<22} {"Suelo":>6} {"Temp":>6} {"HumA":>6} | {"Prediccion":>10}')
print('  ' + '-'*55)

casos = [
    ('Sequia Total',     0,  24, 55),
    ('Suelo Seco',      33,  24, 55),
    ('Suelo Humedo',    51,  24, 60),
    ('Seco + Frio',     30,  10, 80),
    ('Optimo + Niebla', 65,  15, 92),
    ('Saturado + Calor',90,  32, 35),
    ('Critico: estres', 10,  34, 25),
]

for nombre, suelo, temp, hum in casos:
    inp  = np.array([[suelo, temp, hum]], dtype=np.float32)
    pred = float(model.predict(inp, verbose=0)[0][0])
    pred = max(0.0, round(pred, 2))
    if pred < 0.5:   interp = 'No regar'
    elif pred < 4.0: interp = 'Riego suave'
    elif pred < 8.0: interp = 'Riego moderado'
    else:            interp = 'Riego intenso'
    print(f'  {nombre:<22} {suelo:>5}% {temp:>5}C {hum:>5}% | {pred:>8.2f}m  {interp}')

print('=' * 55)

## PASO 3 — Convertir a TFLite

In [ ]:
# Conversion a TFLite — optimizacion DEFAULT (compatible con EloquentTinyML)
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

with open('riego_model.tflite', 'wb') as f:
    f.write(tflite_model)

print(f'Modelo TFLite generado: {len(tflite_model)} bytes ({len(tflite_model)/1024:.1f} KB)')
print('Guardado como riego_model.tflite')

## PASO 4 — Generar modelo_entrenado.h

Este es el archivo que va a `lib/Logic/modelo_entrenado.h` en tu proyecto PlatformIO.

In [ ]:
def generar_header(data, nombre='riego_model'):
    lineas = [
        '// ============================================================',
        '// EcoRiego AI — Modelo compatible con EloquentTinyML',
        '// Arquitectura : Dense(16,relu) -> Dense(8,relu) -> Dense(1,lineal)',
        '// Entradas     : [HumedadSuelo, TemperaturaAire, HumedadAire]',
        '// Salida       : TiempoRiego en minutos [0, 15]',
        f'// Tamano       : {len(data)} bytes',
        '// ============================================================',
        '',
        '#include <cstdint>',
        '',
        f'alignas(8) const unsigned char {nombre}[] = {{',
    ]
    hex_vals = [f'0x{b:02x}' for b in data]
    for i in range(0, len(hex_vals), 12):
        lineas.append('  ' + ', '.join(hex_vals[i:i+12]) + ',')
    lineas += ['};', '', f'const int {nombre}_len = {len(data)};', '']
    return '\n'.join(lineas)

header = generar_header(tflite_model)

with open('modelo_entrenado.h', 'w') as f:
    f.write(header)

print('Archivo modelo_entrenado.h generado correctamente.')
print(f'Tamano del modelo: {len(tflite_model)} bytes')
print()
print('INSTRUCCION:')
print('1. Descarga modelo_entrenado.h desde el panel de archivos de Colab')
print('2. Reemplaza el contenido de lib/Logic/modelo_entrenado.h en tu proyecto')
print('3. Verifica que TENSOR_ARENA_SIZE sea al menos 10 * 1024 en IrrigationBrain.h')
print('4. Compila y sube al ESP32')

In [ ]:
# Descarga automatica en Colab
try:
    from google.colab import files
    files.download('modelo_entrenado.h')
    print('Descarga iniciada automaticamente.')
except:
    print('Descarga manual: panel de archivos de Colab (icono de carpeta a la izquierda)')
    print('Busca el archivo modelo_entrenado.h y haz clic derecho -> Descargar')

## PASO 5 — Ajuste en IrrigationBrain.h

Después de reemplazar el archivo `.h`, verifica que en `lib/Logic/IrrigationBrain.h` el `TENSOR_ARENA_SIZE` sea:

```cpp
#define TENSOR_ARENA_SIZE 10 * 1024
```

El modelo nuevo es más pequeño (arquitectura simple), con 10KB es suficiente y sobra RAM para el resto del sistema.